### snowspark 
Snowpark is a developer framework built by Snowflake that allows you to write and execute non-SQL code—such as Python, Java, and Scala—directly inside Snowflake's processing engine. It eliminates the need to move data out of Snowflake to run complex data engineering pipelines or machine learning models.

 Snowflake Account : [link ](https://app.snowflake.com/ap-southeast-7.aws/ak51667/#/homepage)<br>
- Account name     : mathettu
- password         : Mahee@8143738092
- Account-id       : TAAEZJZ-JP78568
- Role             : ACCOUNTADMIN



In [ ]:
# Creating a Session object 
from snowflake.snowpark import Session 
config= { 
    "account" : "TAAEZJZ-JP78568",
    "user" : "mathettu",
    "password": "Mahee@8143738092",
    "role" :"ACCOUNTADMIN",
    "database": "DEMO_DB",
    "schema": "DEMO_SCHEMA"
}
session=Session.builder.configs(config).create()
print("connection Success ")
print(session.sql("SELECT 1 UNION SELECT 2").collect()[:])


connection Success 
[Row(1=1), Row(1=2)]


Snowpark lazy execution means that Snowflake Snowpark DataFrame operations build a query plan rather than running right away. The actual work is deferred until you trigger a terminal action like collect(), show(), or count(). 
How It Works <br>
**Transformations (Lazy):** Methods like filter(), select(), and join() only add steps to an internal plan. They do not compute data or talk to the server.<br>
**Actions (Eager):** Methods like collect(), show(), write(), and count() force execution. Snowpark compiles the full chain into a SQL query and sends it to Snowflake

In [65]:
# To read the data from the table and selecting the columns and filter command 
from snowflake.snowpark.functions import *
sample_df= session.table('snowflake_sample_data.tpch_sf10.customer')
sample_df=sample_df.select(col("C_CUSTKEY"),col("C_NAME")).filter((col("C_CUSTKEY")<'15010') & (col("C_NAME")=="Customer#000000001"))
sample_df.columns
type(sample_df)

snowflake.snowpark.dataframe.DataFrame

In [22]:
# count of records , describe and ordering the data  
print(sample_df.count())
sample_df.describe().sort(col("SUMMARY").desc()).show()

1500000
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"SUMMARY"  |"C_CUSTKEY"         |"C_NAME"            |"C_ADDRESS"                       |"C_NATIONKEY"      |"C_PHONE"        |"C_ACCTBAL"        |"C_MKTSEGMENT"  |"C_COMMENT"                                         |
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|stddev     |433012.84622976254  |NULL                |NULL                              |7.212353430053189  |NULL             |3176.521993599811  |NULL            |NULL                                                |
|min        |1.0                 |Customer#000000001  |   ,qJqVsHDVWLs6mv6S7Hwh9H        |0.0                |10-100

#### Create DataFrame without Schema Defination

In [29]:
data1=[1,2,3,4]
schema1=['a']
data2=[[1,2,3],[1,2,3],[1,2,3]]
schema2=['a','b','c']
data2=[[1,2,'mahesh'],[1,2,'ramesh'],[1,2,'suresh']]
data3=[[1,2,['2025-10-12']],[1,2,['2025-10-12']],[1,2,['2025-10-12']]]



In [30]:
dataframe=session.create_dataframe(data=data1,schema=schema1)
dataframe=session.create_dataframe(data=data2,schema=schema2)
dataframe1=session.create_dataframe(data=data3,schema=schema2)
dataframe1.show()

------------------------------
|"A"  |"B"  |"C"             |
------------------------------
|1    |2    |[               |
|     |     |  "2025-10-12"  |
|     |     |]               |
|1    |2    |[               |
|     |     |  "2025-10-12"  |
|     |     |]               |
|1    |2    |[               |
|     |     |  "2025-10-12"  |
|     |     |]               |
------------------------------



In [31]:
# results cache to store the data of dataframe into temp table and use 
test=dataframe.cache_result()
test.show()

----------------------
|"A"  |"B"  |"C"     |
----------------------
|1    |2    |mahesh  |
|1    |2    |ramesh  |
|1    |2    |suresh  |
----------------------



#### Creating dataframe with defined schema 

In [66]:
from snowflake.snowpark.types import StructType,StructField,IntegerType,StringType,FloatType,DateType
Schema= StructType(
    [
       StructField("SNO", IntegerType(), False) ,
       StructField("Name", StringType(), False) ,
       StructField("SALARY", FloatType(), False),
       StructField("CR_DATE", DateType(), False)
    ]
)

data =[
    [1,'mahesh',20000.00,'2025-01-24'],
    [2,'suresh',50000.00,'2025-02-24'],
    [3,'ramesh',100000.00,'2025-04-24'],
    [4,'rajesh',200000.00,'2025-05-24']  
]
dataframe=session.create_dataframe(data=data,schema=Schema)
print(type(dataframe))
dataframe.show()


<class 'snowflake.snowpark.dataframe.DataFrame'>
------------------------------------------
|"SNO"  |"NAME"  |"SALARY"  |"CR_DATE"   |
------------------------------------------
|1      |mahesh  |20000.0   |2025-01-24  |
|2      |suresh  |50000.0   |2025-02-24  |
|3      |ramesh  |100000.0  |2025-04-24  |
|4      |rajesh  |200000.0  |2025-05-24  |
------------------------------------------



In [50]:
dataframe.dtypes
dataframe.columns
dataframe.collect()[0]

Row(SNO=1, NAME='mahesh', SALARY=20000.0, CR_DATE=datetime.date(2025, 1, 24))

In [62]:
import pandas as pd 
df=pd.DataFrame(data=([1,2,3,4,5],[3,4,5,6,7]),columns=['a','b','c','d','e'])
# session.create_dataframe(df).show()
type(session.create_dataframe(df))



snowflake.snowpark.table.Table

In [70]:
dataframe.select_expr("SNO,NAME").show()

------------------
|"SNO"  |"NAME"  |
------------------
|1      |mahesh  |
|2      |suresh  |
|3      |ramesh  |
|4      |rajesh  |
------------------



In [93]:
from snowflake.snowpark.types import StructType,StructField,StringType,DateType
schema= StructType(
    [
        StructField("FIRST_NAME",StringType()),
        StructField("LAST_NAME",StringType()),
        StructField("EMAIL",StringType()),
        StructField("ADDRESS",StringType()),
        StructField("CITY",StringType()),
        StructField("DOJ",DateType())

    ]
)
session.sql("USE SCHEMA DEMO_DB.DEMO_SCHEMA")
df=session.read.options({"ON_ERROR":"SKIP_FILE_7"}).schema(schema).csv('@my_s3_stage/employee/')
df.count()
# df=df.select("FIRST_NAME","LAST_NAME").filter(col("FIRST_NAME")=='Nyssa')
# df.show()
df.show()
df.write.mode('append').saveAsTable("snow_csv")


--------------------------------------------------------------------------------------------------------------
|"FIRST_NAME"  |"LAST_NAME"  |"EMAIL"                   |"ADDRESS"               |"CITY"        |"DOJ"       |
--------------------------------------------------------------------------------------------------------------
|Lem           |Boissier     |lboissier@sf_tuts.com     |3002 Ruskin Trail       |Shikārpur     |2017-08-25  |
|Iain          |Hanks        |ihanks1@sf_tuts.com       |2 Pankratz Hill         |Monte-Carlo   |2017-12-10  |
|Avo           |Laudham      |alaudham2@sf_tuts.com     |6948 Debs Park          |Prażmów       |2017-10-18  |
|Emili         |Cornner      |ecornner3@sf_tuts.com     |177 Magdeline Avenue    |Norrköping    |2017-08-13  |
|Harrietta     |Goolding     |hgoolding4@sf_tuts.com    |450 Heath Trail         |Osielsko      |2017-11-27  |
|Arlene        |Davidovits   |adavidovitsk@sf_tuts.com  |7571 New Castle Circle  |Meniko        |2017-05-03  |
|